# 15 · Redes Neuronales de Reorganización Autónoma (RNRA)

**Estado: sin validar.** Una red con ranuras de neuronas y máscaras de conexiones que, mientras aprende, **poda** neuronas redundantes, **activa** neuronas de reserva solo cuando falta capacidad y **reasigna conexiones** hacia las regiones del problema que más lo necesitan, sin pasar de un presupuesto de conexiones.

Lo nuevo que se intenta probar no es crecer, podar ni mover conexiones (eso ya lo hacen Net2Net, SET, RigL y Structured RigL): es **transferir capacidad desde lo redundante hacia las regiones difíciles** y **comprobar con una prueba corta, con y sin el cambio, que de verdad ayuda** antes de aceptarlo.

**Hipótesis (deben poder refutarse)**
- **H1** eficiencia estructural: igual o mayor precisión que una red fija con presupuesto equivalente.
- **H2** plasticidad sin crecimiento ilimitado: aprende tareas nuevas olvidando menos, sin que crezcan los parámetros.
- **H3** especialización emergente: grupos de neuronas se reparten regiones sin asignarlas a mano.
- **H4** estabilidad: la histéresis evita ciclos de crear y quitar la misma neurona.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo* (sirve CPU; no necesita GPU). Cada sección dice cuánto tarda. Los umbrales de éxito se fijaron **antes** de correr y están en `laboratorio/rnra/experimentos.py` (`criterios`). Un resultado negativo también vale: se anota igual.

**Límites de la v0:** los pesos se guardan con el tamaño máximo y se enmascaran, así que apagar conexiones **no** ahorra tiempo ni memoria reales; el coste que se mide son los FLOPs lógicos de las conexiones activas (incluida la prueba de cada cambio). Las neuronas nuevas eligen sus conexiones al azar. La novedad frente a RigL y Structured RigL todavía requiere una revisión bibliográfica más exhaustiva.

## 1. Preparar (≈ 2 min)

In [ ]:
import os, subprocess
from google.colab import userdata

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from dataclasses import replace
from laboratorio.banco.tareas import cargar_tarea
from laboratorio.rnra import experimentos as ex
from laboratorio.rnra.controlador import ABLACIONES, COMPARADORES, Config, Entrenador, entrenar

SEMILLAS = range(10)                  # 10 semillas (baja a 3 para probar)
CFG = Config()                        # 800 pasos; presupuesto = conexiones de un MLP denso de 16 neuronas por capa
BASE = [*COMPARADORES, "rnra", "rnra_aleatorio"]
print(CFG)

## 2. E1 · ¿Detecta cuándo falta capacidad? (≈ 3 min)

XOR, círculos y lunas. Pregunta: ¿crece solo cuando hace falta y con qué frecuencia acepta el cambio?

In [ ]:
for nombre in ex.ESTUDIOS["E1"]:
    t = cargar_tarea(nombre)
    res = ex.correr(t, BASE, SEMILLAS, CFG)
    print(f"\n### {nombre}\n"); print(ex.tabla(res)); print(); print(ex.tabla_comparacion(res))
    print(ex.criterios(res))
    ev = [e for r in res["rnra"] for e in r.eventos]
    print(f"eventos de RNRA: {len(ev)} · aceptados {sum(e['aceptado'] for e in ev)} · neuronas podadas {sum(e['podadas'] for e in ev)} · activadas {sum(e['activadas'] for e in ev)}")

## 3. E2 · ¿Redistribuye hacia las regiones difíciles? (≈ 5 min)

Tarea caótica de 20 dimensiones. Además de la precisión, se mide **dónde** se concentra la capacidad: la demanda `D_r` por región y la especialización de las neuronas (0 = reparten igual, 1 = viven en una sola región), contra el control de reorganización aleatoria.

In [ ]:
t = cargar_tarea("caotica_20d")
res2 = ex.correr(t, BASE, SEMILLAS, CFG)
print(ex.tabla(res2)); print(); print(ex.tabla_comparacion(res2)); print(ex.criterios(res2))

esp = {}
for m in ("rnra", "rnra_aleatorio", "rigl"):
    vals = []
    for s in list(SEMILLAS)[:5]:
        e = Entrenador(m, t.entradas, t.salidas, t.tipo, CFG, s); e.fijar_datos(t.x_entreno, t.y_entreno); e.entrenar_tarea()
        vals.append(ex.especializacion(e, e.x))
    esp[m] = vals
print({m: f"{np.mean(v):.3f} ± {np.std(v):.3f}" for m, v in esp.items()})

e = Entrenador("rnra", t.entradas, t.salidas, t.tipo, CFG, 0); e.fijar_datos(t.x_entreno, t.y_entreno); e.entrenar_tarea()
plt.bar(range(e.dem.k), e.dem.valor().numpy()); plt.xlabel("región"); plt.ylabel("demanda D_r (pérdida media)"); plt.title("Dificultad por región al final"); plt.show()

## 4. E3 · Cambios de distribución controlados (≈ 2 min)

Dos lunas que se giran 0°, 60° y 120°. Se mide la exactitud final media en las tres tareas y el **olvido** (cuánto valía cada tarea al aprenderla menos cuánto vale al final).

In [ ]:
sec = ex.lunas_rotadas()
res3 = ex.correr_continuo(sec, BASE, SEMILLAS, CFG)
for m, rs in res3.items():
    print(f"{m:24s} final {np.mean([r.metrica_prueba for r in rs])*100:5.1f} %   olvido {np.mean([r.perdida_prueba for r in rs])*100:5.1f} %")
print(ex.tabla_comparacion(res3))

## 5. E4 · Aprendizaje continuo (≈ 6 min con dígitos; ≈ 30 min con MNIST)

Dígitos permutados (misma tarea, píxeles mezclados), dígitos partidos (cinco tareas de dos clases, una cabeza por tarea) y, si activas `USAR_MNIST`, **Permuted MNIST** y **Split MNIST** de Hugging Face. RNRA protege las neuronas que fueron importantes en tareas anteriores (término `λ·L_olvido` simplificado). Los comparadores no tienen ninguna protección: es la comparación del sistema completo, no de un solo mecanismo.

In [ ]:
USAR_MNIST = False
for sec in (ex.digitos_permutados(), ex.digitos_partidos()):
    res4 = ex.correr_continuo(sec, BASE, SEMILLAS, CFG)
    print(f"\n### {sec.nombre}")
    for m, rs in res4.items():
        print(f"{m:24s} final {np.mean([r.metrica_prueba for r in rs])*100:5.1f} %   olvido {np.mean([r.perdida_prueba for r in rs])*100:5.1f} %")
    print(ex.tabla_comparacion(res4))

if USAR_MNIST:
    from laboratorio.banco.tareas import cargar_tarea as _c
    mn = _c("mnist")
    perm = np.random.default_rng(0)
    tareas = []
    for k in range(5):
        p = np.arange(784) if k == 0 else perm.permutation(784)
        tareas.append(ex.Subtarea(mn.x_entreno.reshape(len(mn.x_entreno), -1)[:, p], mn.y_entreno, mn.x_prueba.reshape(len(mn.x_prueba), -1)[:, p], mn.y_prueba))
    pm = ex.Secuencia("permuted_mnist", tareas, 784, 10)
    resm = ex.correr_continuo(pm, BASE, range(5), replace(CFG, pasos=1500, ancho_pequeno=64, activas_inicial=96, ancho_max=192, ancho_grande=192))
    for m, rs in resm.items():
        print(f"{m:24s} final {np.mean([r.metrica_prueba for r in rs])*100:5.1f} %   olvido {np.mean([r.perdida_prueba for r in rs])*100:5.1f} %")

## 6. E5 · Fuera de lo sintético: Fashion-MNIST (≈ 15 min, baja datos de Hugging Face)

Para 784 entradas el presupuesto y las neuronas de reserva se escalan (`ancho_pequeno=64`).

In [ ]:
t5 = cargar_tarea("fashion_mnist")
cfg5 = replace(CFG, pasos=1500, ancho_pequeno=64, activas_inicial=96, ancho_max=192, ancho_grande=192, lote=256)
res5 = ex.correr(t5, BASE, range(5), cfg5)
print(ex.tabla(res5)); print(); print(ex.tabla_comparacion(res5)); print(ex.criterios(res5))

## 7. Ablaciones: ¿qué mecanismo produce la mejora? (≈ 8 min)

`rnra_sin_utilidad`, `sin_redundancia`, `sin_local` (sin demanda por región), `sin_histeresis`, `sin_crecimiento`, `sin_poda`, más el control `rnra_aleatorio`. Se corre en la tarea caótica.

In [ ]:
t = cargar_tarea("caotica_20d")
resa = ex.correr(t, list(ABLACIONES), SEMILLAS, CFG)
print(ex.tabla(resa))
plt.figure(figsize=(7, 3)); ms = list(resa)
plt.barh(ms, [np.mean([r.metrica_prueba for r in resa[m]]) * 100 for m in ms]); plt.xlim(60, 90); plt.xlabel("exactitud de prueba (%)"); plt.show()

## 8. Veredicto contra los umbrales

Los criterios están en `experimentos.criterios`: +2 puntos de precisión sobre el mejor comparador con el mismo presupuesto (o ventaja de eficiencia), parámetros activos iguales o menores, 20 % menos FLOPs para llegar a la precisión objetivo, cero ciclos estructurales y, para decir que aporta algo nuevo, ventaja **en al menos dos familias de tareas**. Con `p (Holm)` < 0,05 en las comparaciones de arriba. Si no se cumplen, se escribe como resultado negativo en el informe.

In [ ]:
for nombre, r, clasif in [("caotica_20d", res2, True)]:
    c = ex.criterios(r, mayor_mejor=clasif)
    print(nombre, {k: v["ok"] for k, v in c.items() if isinstance(v, dict)}, "· mejor comparador:", c["mejor_comparador"])

## 9. Guardar el informe en Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/Notbook/resultados_rnra", exist_ok=True)
!python -m laboratorio.rnra --todo --ablaciones --semillas 10 --salida /content/drive/MyDrive/Notbook/resultados_rnra/15-reorganizacion-autonoma.md > /dev/null
print("Guardado en Drive: Notbook/resultados_rnra/")